# Recolección de telemetría de tráfico

Usá esta notebook para convertir un video en **telemetría cruda por minuto** y un video anotado. No entrena ni clasifica estados.

| Preset | Cuándo usarlo | Efecto externo |
|---|---|---|
| `CollectionPreset.LOCAL` | Recolección habitual | Archivos en el runtime |
| `CollectionPreset.POSTGRES` | Conservar telemetría raw | Escritura con perfil `collection` |
| `CollectionPreset.TRACKING_DIAGNOSTIC` | Revisar tracking y señales | HUD técnico; sin PostgreSQL |
| `CollectionPreset.CUSTOM` | Ajustar opciones puntuales | Depende de la configuración final |

**Inicio rápido recomendado:** conservá `CollectionPreset.LOCAL`, ejecutá `Run All` y subí el MP4 cuando aparezca el selector. La carga del video siempre está disponible en Colab; `download_outputs` sólo decide si los resultados se descargan después. Se necesitan 60 segundos para producir la primera fila.

<details>
<summary><strong>Personalización tipada</strong></summary>

Seleccioná `CUSTOM` y construí `CUSTOM_CONFIG` con `replace(collection_preset_config(...), ...)`. Las descargas, PostgreSQL, HUD técnico y planes privados siguen siendo opt-in. Consultá la [guía central de presets](../../../docs/operations/notebook-configuration.md).

</details>

> Si abriste esta notebook antes de una actualización, volvé a abrirla desde el enlace de GitHub. `git pull` actualiza el checkout, pero no reemplaza las celdas que ya están abiertas en Colab.


In [ ]:
# Preparación del entorno: ejecutá esta celda una vez por runtime.
import importlib.util
import os
import runpy
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORKSPACE_DIR = next(
        (
            path
            for path in candidates
            if (path / "vaaet-core/pyproject.toml").is_file()
            and (path / "vaaet-persistence/pyproject.toml").is_file()
            and (path / "vaaet-ml/pyproject.toml").is_file()
        ),
        None,
    )
    if WORKSPACE_DIR is None:
        raise RuntimeError("No se encontró el workspace VAAET con core y ML.")
CORE_ROOT = WORKSPACE_DIR / "vaaet-core"
PERSISTENCE_ROOT = WORKSPACE_DIR / "vaaet-persistence"
ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
REPO_ROOT = ML_ROOT
os.chdir(ML_ROOT)
BOOTSTRAP = runpy.run_path(str(ML_ROOT / "scripts" / "notebook_bootstrap.py"))
RUNTIME = BOOTSTRAP["bootstrap_notebook"](
    workspace_root=WORKSPACE_DIR,
    core_root=CORE_ROOT,
    persistence_root=PERSISTENCE_ROOT,
    ml_root=ML_ROOT,
    core_extras=('vision',),
    ml_extras=('database',),
    in_colab=IN_COLAB,
    framework='torch',
    require_gpu=True,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
VAAET_ML_PACKAGE_FILE = RUNTIME.ml_package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Configuración del workflow: editá únicamente esta celda.
from dataclasses import replace

from vaaet_ml.workflow_presets import (
    CollectionPreset,
    collection_preset_config,
    render_workflow_summary,
    resolve_collection_config,
)

SELECTED_PRESET = CollectionPreset.LOCAL
CUSTOM_CONFIG = None
WORKFLOW_CONFIG = resolve_collection_config(
    SELECTED_PRESET,
    custom_config=CUSTOM_CONFIG,
)

print(render_workflow_summary(SELECTED_PRESET, WORKFLOW_CONFIG))


In [ ]:
# Imports del workflow: no edites esta celda.
import cv2
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import torch
import ultralytics

from importlib.metadata import version as package_version

from vaaet_persistence import PipelineRunMetadata, PipelineWorkflow, database_engine, inspect_database, persist_raw_telemetry, pipeline_run
from vaaet_ml.data.database import DatabaseProfile, get_optional_database_settings
from vaaet_ml.data.datasets import merge_raw_telemetry_csv
from vaaet_ml.notebook_io import resolve_video_input
from vaaet.logging import configure_logging
from vaaet_ml.view_plan import load_video_view_plan
from vaaet.vision.analysis import analyze_video
from vaaet.vision.hud import HudConfig

configure_logging()
VIEW_PLAN = load_video_view_plan(WORKFLOW_CONFIG.view_plan_path)


## 1. Seleccionar el video

En Colab se abrirá el selector de archivos aunque las descargas estén desactivadas. Si el MP4 ya está en `/content`, asigná su ruta a `EXPLICIT_VIDEO_PATH` y no se volverá a cargar. En local se usa únicamente el ejemplo declarado en `data/sample/`.

**Seleccionar/subir** incorpora el MP4 al runtime. **Generar** crea el video anotado y el CSV dentro del runtime. **Descargar** los copia a tu equipo sólo si `download_outputs=True`. **Persistir** escribe la telemetría en PostgreSQL únicamente con el preset correspondiente.

Ejemplos: 16 segundos producen sólo video; 60 segundos producen una fila; 125 segundos producen dos filas y descartan los 5 segundos finales.

In [ ]:
EXPLICIT_VIDEO_PATH: Path | None = None
_video_uploader = None
if IN_COLAB:
    from google.colab import files

    _video_uploader = files.upload

VIDEO_PATH = resolve_video_input(
    EXPLICIT_VIDEO_PATH,
    in_colab=IN_COLAB,
    uploader=_video_uploader,
    staging_directory=Path("/content") if IN_COLAB else REPO_ROOT / "data/sample",
    local_fallback=REPO_ROOT / "data/sample/sample.mp4",
)
result = None
df_raw = None
COLLECTION_PIPELINE_RUN_ID = None
PROCESSED_VIDEO_PATH = None

print(f"✅ Video seleccionado: {VIDEO_PATH}")
print("➡️ Siguiente paso: procesá el video.")

## 2. Procesar el video

YOLO descarga sus pesos oficiales la primera vez; no se guardan en Git. Durante el proceso verás detecciones, velocidades aproximadas y conteos acumulados.

Al terminar, la celda muestra cuántos minutos completos generó, cuánto tiempo parcial descartó y dónde quedaron el video y el CSV.

In [ ]:
result = None
df_raw = None
COLLECTION_PIPELINE_RUN_ID = None
PROCESSED_VIDEO_PATH = None
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Seleccioná o subí un MP4 válido antes de continuar.")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_annotated.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_annotated.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.COLLECTION, application_name="vaaet-ml-collection", application_version=package_version("vaaet-ml"), git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem)
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    result = analyze_video(
        VIDEO_PATH, OUTPUT_VIDEO, hud_config=HudConfig(debug=WORKFLOW_CONFIG.hud_debug), view_plan=VIEW_PLAN
    )
    _run.set_output_rows(len(result.telemetry))
COLLECTION_PIPELINE_RUN_ID = str(_run.id)
PROCESSED_VIDEO_PATH = VIDEO_PATH.resolve()

RAW_CSV = REPO_ROOT / "data/raw/traffic_data_raw.csv"
display(result.telemetry)
print(f"✅ Video anotado: {result.video_path}")
if result.telemetry.empty:
    print("ℹ️ El video se procesó bien, pero no contiene un minuto completo.")
    print("   No se generará telemetría, no se actualizará el CSV y PostgreSQL se omitirá.")
    print(f"   Duración procesada: {result.processed_duration_seconds:.1f}s | mínimo: 60.0s")
else:
    df_raw = merge_raw_telemetry_csv(result.telemetry, RAW_CSV)
    print(f"✅ Minutos completos: {result.complete_minutes} | tramo final descartado: {result.discarded_partial_seconds:.1f}s")
    print(f"✅ CSV canónico: {RAW_CSV} ({len(df_raw)} filas únicas)")
print("➡️ Siguiente paso: revisá la persistencia opcional.")

if IN_COLAB and WORKFLOW_CONFIG.download_outputs:
    from google.colab import files

    files.download(str(result.video_path))
    if not result.telemetry.empty and RAW_CSV.is_file():
        files.download(str(RAW_CSV))
if IN_COLAB and not WORKFLOW_CONFIG.download_outputs:
    print(f"ℹ️ Descargas desactivadas; video disponible en {result.video_path}")
    if RAW_CSV.is_file():
        print(f"ℹ️ CSV disponible en {RAW_CSV}")

## 3. Guardar en PostgreSQL (opcional)

Esta etapa se omite por defecto. Si la activaste, usa únicamente el perfil `collection` y escribe en `vaaet_raw`. Repetir el mismo clip no duplica filas porque `(clip_id, record_time)` identifica cada minuto.

Las credenciales se configuran según la [guía canónica de Colab](../../../docs/operations/colab-guide.md#secrets-y-postgresql); tenerlas disponibles nunca activa la escritura por sí solo.

In [ ]:
if result is None or COLLECTION_PIPELINE_RUN_ID is None or PROCESSED_VIDEO_PATH != VIDEO_PATH.resolve():
    print("ℹ️ PostgreSQL omitido: no existe un análisis nuevo y completo para el video seleccionado.")
elif result.telemetry.empty:
    print("ℹ️ PostgreSQL omitido: el video no tiene un minuto completo.")
elif WORKFLOW_CONFIG.persist_to_database:
    settings = get_optional_database_settings(DatabaseProfile.COLLECTION)
    if settings is None:
        raise RuntimeError("Activaste PostgreSQL, pero el perfil collection no está configurado.")
    with database_engine(settings) as db_engine:
        health = inspect_database(db_engine, DatabaseProfile.COLLECTION)
        print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
        with pipeline_run(_run_metadata, engine=db_engine, run_id=COLLECTION_PIPELINE_RUN_ID) as _db_run:
            inserted = persist_raw_telemetry(
                result.telemetry, engine=db_engine, pipeline_run_id=_db_run.id
            )
            _db_run.set_output_rows(inserted)
    print(f"✅ PostgreSQL: {inserted} filas nuevas en vaaet_raw.traffic_data | run={COLLECTION_PIPELINE_RUN_ID}")
else:
    print("ℹ️ PostgreSQL desactivado; el video y el CSV siguen disponibles.")
print("✅ Flujo de recolección terminado.")